In [1]:
import wandb
import matplotlib as mpl
import matplotlib.pyplot as plt
import collections
import numpy as np
import pandas as pd

api = wandb.Api(timeout=39)

# Define your project and entity (replace with your specific values)
entity = "wang-kevin3290-princeton-university"  # e.g., your username or team name
project = "clean_JaxGCRL_test"


In [2]:
env_id = "arm_binpick_hard"
eval_env_id = None #"ant_u5_maze_eval"
env_title = "Arm Binpick Hard"
num_epochs = 100
metric = 'eval/episode_success'
metric_label = "Success"
# valid_depths = [8, 16]
valid_depths = [4, 64]
valid_widths = [128, 256, 512, 1024, 2048, 4096]

filters = {
    "config.env_id": env_id,
    "config.eval_env_id": eval_env_id if eval_env_id else {"$in": ['', None, env_id]}, #{"$ne": "xxx"}, #
    "config.num_envs": 512,
    "config.num_sgd_batches_per_training_step": 800,
    "config.critic_network_width": 256,
    "config.batch_size": 512,
    "config.num_epochs": {"$in": [num_epochs]}, #num_epochs,
    "config.disable_entropy": {"$ne": 1},
    "config.use_relu": {"$ne": 1},
    "config.resnet": "noishmistake4_nodense",
    "config.dropout": None,
    "config.logsumexp_penalty_coeff": {"$in": [None, 0.1]},
}

runs_api = api.runs(path=f"{entity}/{project}", filters=filters) #runs filtered by wandb api
len(runs_api)

169

In [3]:
#local filters
runs = []
dfs = {} #scratch space for arm_binpick_easy


for run in runs_api:
    # check it actually has num_epoch steps completed
    if '_step' not in run.summary.keys():
        continue
    if run.summary['_step'] != run.config['num_epochs'] - 1:
        continue

    #check actor_depth == critic_depth and filter for valid_depths
    actor_depth = run.config['actor_depth']
    critic_depth = run.config['critic_depth']
    
    if actor_depth != critic_depth:
        continue
    if actor_depth not in valid_depths:
        continue

    runs.append(run)
len(runs)

29

In [4]:
depth_count = collections.defaultdict(int)
for run in runs:
    # print(f"{run.config['actor_depth']}: {run.config['seed']}")
    depth_count[run.config['actor_depth']] += 1

for d, c in depth_count.items():
    print(f"{d}: {c}")

4: 7
64: 22


In [5]:
# Gather the raw metric curves by depth
run_vals = collections.defaultdict(list) #{d: [] for d in valid_depths}, where it's a list of lists

for run in runs:
    depth = run.config['actor_depth']
    # Pull the full training history from W&B
    hist = run.history()
    if metric not in hist.columns:
        print("WARNING: metric not found in hist.columns for depth {depth} seed {run.config['seed']}, continuing...")
        continue
    last_values = hist[metric].values[-5:]  # shape = (num_epochs,)
    avg = sum(last_values) / len(last_values)
    run_vals[depth].append(avg)

avgs = {}

for depth in run_vals.keys():
    avgs[depth] = sum(run_vals[depth]) / len(run_vals[depth])

avgs

{4: 38.22611607142857, 64: 218.6947443181818}

In [26]:
def get_avgs(runs_api):
    #local filters
    runs = []
    dfs = {} #scratch space for arm_binpick_easy


    for run in runs_api:
        # check it actually has num_epoch steps completed
        if '_step' not in run.summary.keys():
            continue
        if run.summary['_step'] != run.config['num_epochs'] - 1:
            continue

        #check actor_depth == critic_depth and filter for valid_depths
        actor_depth = run.config['actor_depth']
        critic_depth = run.config['critic_depth']

        if actor_depth != critic_depth:
            continue
        if actor_depth not in valid_depths:
            continue

        runs.append(run)
    len(runs)
    
    depth_count = collections.defaultdict(int)
    for run in runs:
        # print(f"{run.config['actor_depth']}: {run.config['seed']}")
        depth_count[run.config['actor_depth']] += 1

    for d, c in depth_count.items():
        print(f"{d}: {c}")
        
    # Gather the raw metric curves by depth
    depth_curves = collections.defaultdict(list) #{d: [] for d in valid_depths}, where it's a list of lists

    for run in runs:
        depth = run.config['actor_depth']
        # Pull the full training history from W&B
        hist = run.history()
        if metric not in hist.columns:
            print("WARNING: metric not found in hist.columns for depth {depth} seed {run.config['seed']}, continuing...")
            continue
        values = hist[metric].values  # shape = (num_epochs,)
        depth_curves[depth].append(values)


    # Now, for each depth, compute mean/std
    depth_last5_avg = {}  # Dictionary to store depth -> average of last 5 values

    for depth in valid_depths:
        curves = depth_curves[depth]
        

        # Stack all runs for this depth: shape = (num_runs, num_epochs)
        arr = np.stack(curves, axis=0)  # e.g. (N_runs, 100)
        mean_last5 = arr[:, -5:].mean(axis=1)  # Average of last 5 values for each curve
        depth_last5_avg[depth] = mean_last5.mean()  # Overall average of last 5 values
        print(f"{len(curves)} curves for depth {depth}. Avg: {depth_last5_avg[depth]}")

    return depth_last5_avg

In [27]:
def getRuns(env_id, eval_env_id, env_title, num_epochs, metric, metric_label, valid_depths):
    filters = {
        "config.env_id": env_id,
        "config.eval_env_id": eval_env_id if eval_env_id else {"$in": ['', None, env_id]}, #{"$ne": "xxx"}, #
        "config.num_envs": 512,
        "config.num_sgd_batches_per_training_step": 800,
        "config.critic_network_width": 256,
        "config.batch_size": 512,
        "config.num_epochs": {"$in": [num_epochs]}, #num_epochs,
        "config.disable_entropy": {"$ne": 1},
        "config.use_relu": {"$ne": 1},
        "config.resnet": "noishmistake4_nodense",
        "config.dropout": None,
        "config.logsumexp_penalty_coeff": {"$in": [None, 0.1]},
    }

    runs_api = api.runs(path=f"{entity}/{project}", filters=filters) #runs filtered by wandb api
    return runs_api

In [29]:
allData = {}

In [30]:
env_id = "arm_push_easy"
eval_env_id = None #"ant_u5_maze_eval"
num_epochs = 100
runs_api = getRuns(env_id, eval_env_id, env_title, num_epochs, metric, metric_label, valid_depths)
len(runs_api)
allData[env_id] = get_avgs(runs_api)
print(allData[env_id])

4: 7
64: 12
7 curves for depth 4
12 curves for depth 64
{4: 307.5258928571429, 64: 761.7475260416667}


In [31]:
env_id = "ant_big_maze"
eval_env_id = "ant_big_maze_eval"
num_epochs = 100
runs_api = getRuns(env_id, eval_env_id, env_title, num_epochs, metric, metric_label, valid_depths)
len(runs_api)
allData[env_id] = get_avgs(runs_api)
print(allData[env_id])

4: 10
64: 5
10 curves for depth 4
5 curves for depth 64
{4: 60.723749999999995, 64: 440.8053125}


In [32]:
env_id = "arm_binpick_hard"
eval_env_id = None #"ant_u5_maze_eval"
num_epochs = 100
env_title = "Arm Binpick Hard"
metric = 'eval/episode_success'
metric_label = "Success"


runs_api = getRuns(env_id, eval_env_id, env_title, num_epochs, metric, metric_label, valid_depths)
len(runs_api)
allData[env_id] = get_avgs(runs_api)
# print(allData[env_id])

4: 7
64: 22
7 curves for depth 4
22 curves for depth 64


In [33]:
env_id = "arm_binpick_easy"
eval_env_id = None #"ant_u5_maze_eval"
num_epochs = 100
runs_api = getRuns(env_id, eval_env_id, env_title, num_epochs, metric, metric_label, valid_depths)
len(runs_api)
allData[env_id] = get_avgs(runs_api)
# print(allData[env_id])

4: 7
64: 5
7 curves for depth 4
5 curves for depth 64


In [34]:
env_id = "arm_push_hard"
eval_env_id = None #"ant_u5_maze_eval"
num_epochs = 100
runs_api = getRuns(env_id, eval_env_id, env_title, num_epochs, metric, metric_label, valid_depths)
len(runs_api)
allData[env_id] = get_avgs(runs_api)
# print(allData[env_id])

4: 7
64: 12
7 curves for depth 4
12 curves for depth 64


In [35]:
env_id = "ant_u5_maze"
eval_env_id = "ant_u5_maze_eval"
# print(allData[env_id][0][4])
num_epochs = 400
runs_api = getRuns(env_id, eval_env_id, env_title, num_epochs, metric, metric_label, valid_depths)
len(runs_api)
allData[env_id] = get_avgs(runs_api)
# print(allData[env_id])

4: 39
64: 8
39 curves for depth 4
8 curves for depth 64


In [36]:
env_id = "ant_u4_maze"
eval_env_id = "ant_u4_maze_eval"
num_epochs = 100
runs_api = getRuns(env_id, eval_env_id, env_title, num_epochs, metric, metric_label, valid_depths)
len(runs_api)
allData[env_id] = get_avgs(runs_api)
# print(allData[env_id])

4: 25
64: 12
25 curves for depth 4
12 curves for depth 64


In [37]:
env_id = "ant_hardest_maze"
eval_env_id = None #"ant_u5_maze_eval"
num_epochs = 200
runs_api = getRuns(env_id, eval_env_id, env_title, num_epochs, metric, metric_label, valid_depths)
len(runs_api)
allData[env_id] = get_avgs(runs_api)
# print(allData[env_id])

64: 5
4: 7
7 curves for depth 4
5 curves for depth 64


In [38]:
env_id = "humanoid_big_maze"
eval_env_id = None #"humanoid_big_maze_eval"
num_epochs = 400
runs_api = getRuns(env_id, eval_env_id, env_title, num_epochs, metric, metric_label, valid_depths)
len(runs_api)
allData[env_id] = get_avgs(runs_api)
# print(allData[env_id])

64: 7
4: 25
25 curves for depth 4
7 curves for depth 64


In [39]:
# data = [0 for i in range(400)]
# print(allData["humanoid_big_maze"][0][4])

In [40]:
env_id = "humanoid_u_maze"
eval_env_id = None #"humanoid_big_maze_eval"
num_epochs = 400
runs_api = getRuns(env_id, eval_env_id, env_title, num_epochs, metric, metric_label, valid_depths)
len(runs_api)
allData[env_id] = get_avgs(runs_api)
# print(allData[env_id])

4: 5
64: 5
5 curves for depth 4
5 curves for depth 64


In [41]:
allData

{'arm_push_easy': {4: 307.5258928571429, 64: 761.7475260416667},
 'ant_big_maze': {4: 60.723749999999995, 64: 440.8053125},
 'arm_binpick_hard': {4: 38.22611607142857, 64: 218.69474431818182},
 'arm_binpick_easy': {4: 157.07633928571428, 64: 383.02812499999993},
 'arm_push_hard': {4: 171.0504464285714, 64: 410.1154947916666},
 'ant_u5_maze': {4: 0.9762019230769232, 64: 61.164453125},
 'ant_u4_maze': {4: 11.366999999999997, 64: 285.61341145833336},
 'ant_hardest_maze': {4: 214.55602678571427, 64: 387.06374999999997},
 'humanoid_big_maze': {4: 0.0050625, 64: 58.62388392857142},
 'humanoid_u_maze': {4: 3.1525, 64: 158.85249999999996}}

In [42]:
# allData

In [43]:
import pandas as pd
df = pd.DataFrame(allData).T
df.columns = valid_depths  # Ensure columns are ordered as required
df = df[valid_depths]  # Ensure consistent ordering of columns
df

,4,64
arm_push_easy,307.525893,761.747526
ant_big_maze,60.723750,440.805313
arm_binpick_hard,38.226116,218.694744
arm_binpick_easy,157.076339,383.028125
arm_push_hard,171.050446,410.115495
ant_u5_maze,0.976202,61.164453
ant_u4_maze,11.367000,285.613411
ant_hardest_maze,214.556027,387.063750
humanoid_big_maze,0.005063,58.623884
humanoid_u_maze,3.152500,158.852500


In [45]:
df['4 --> 64'] = df[64] / df[4]
df

,4,64,4 --> 64
arm_push_easy,307.525893,761.747526,2.477019
ant_big_maze,60.723750,440.805313,7.259191
arm_binpick_hard,38.226116,218.694744,5.721082
arm_binpick_easy,157.076339,383.028125,2.438484
arm_push_hard,171.050446,410.115495,2.397629
ant_u5_maze,0.976202,61.164453,62.655534
ant_u4_maze,11.367000,285.613411,25.126543
ant_hardest_maze,214.556027,387.063750,1.804022
humanoid_big_maze,0.005063,58.623884,11580.026455
humanoid_u_maze,3.152500,158.852500,50.389374
